# VITS

A self-contained refresher on **VITS** (*Variational Inference with adversarial learning for end-to-end Text-to-Speech*) — the model that collapses the classic text→mel→vocoder TTS pipeline into a **single** network that goes straight from text to waveform.

**Domain:** Speech & Audio  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**VITS** (Kim, Kong & Son, 2021) is a **fully end-to-end** neural text-to-speech model: one network maps characters/phonemes **directly to a raw waveform**, with no separately trained acoustic model + vocoder. It is built as a **conditional variational autoencoder (CVAE)** whose prior is shaped by **normalizing flows**, trained with an **adversarial (GAN) loss** on the generated audio, and aligned to text at training time by **Monotonic Alignment Search (MAS)**. A **stochastic duration predictor** lets the same sentence be spoken with different, natural-sounding rhythms.

**The problem it solves.** The dominant TTS recipe ([[tacotron]] → vocoder, FastSpeech → vocoder) is **two-stage**: an acoustic model predicts a mel spectrogram, then a *separately trained* vocoder turns mel → audio. That seam causes the classic failure mode — the vocoder is trained on *ground-truth* mels but sees the acoustic model's *imperfect* predicted mels at inference (a train/inference mismatch), so quality drops and you must fine-tune or carefully match feature configs. VITS removes the mel bottleneck and the second model entirely: it trains the whole thing jointly against the waveform, reaching **MOS on par with ground-truth recordings** while being **non-autoregressive** (fast, parallel synthesis).

**Reach for it when** you want high-quality open-source TTS in one model, fast parallel inference, natural prosody variation, and a single training pipeline — it's the backbone of Coqui's best models, 🤗 `VitsModel`, and Meta's MMS-TTS for 1000+ languages.

**Skip it when** you need zero-shot voice cloning from a few seconds of audio (reach for [[styletts2]], [[bark]], or [[elevenlabs]]), fine-grained style/emotion control, or you just want a turnkey cloud API without training/hosting ([[azure-speech]], [[amazon-polly]]).

## 2. Mental Model

**One model with two doors into the same latent space — trained together, used one-way.**

```
  TRAIN (both doors open, learn a shared latent z):

      text ─► [text encoder] ─► prior  p(z|text) ──┐
                                  ▲  (flow f_θ)     │  pulled together
                                  │                 ▼     by the
     audio ─► [posterior enc] ─► q(z|audio) ───► [ z ] ─► [decoder/HiFi-GAN] ─► waveform ─► GAN
                (from linear spectrogram)                 (vocoder built in)
                                  ▲
                          MAS aligns text↔audio frames (monotonic, hard)

  INFER (text door only):

      text ─► encoder ─► sample z ~ prior ─► flow⁻¹ ─► decoder ─► waveform
                 │
       stochastic duration predictor → how many frames each token gets
```

- During **training** the model sees both text *and* the real audio. A **posterior encoder** reads the audio (as a linear spectrogram) and produces a latent `z`; the **decoder** (a HiFi-GAN generator) reconstructs the waveform from `z`. Simultaneously the **text encoder + normalizing flow** learn a **prior** over the *same* `z` that depends only on text. **MAS** decides which text token each audio frame belongs to.
- At **inference** the audio door is gone: you sample `z` from the text-conditioned prior, push it through the flow's inverse, and the decoder emits the waveform. **No mel spectrogram, no second model.**
- The **stochastic duration predictor** is what makes it sound human: instead of one fixed duration per token, it *samples* durations, so repeated synthesis of the same text varies its rhythm — a one-to-many mapping that deterministic models can't capture.

Think of `z` as a **secret meeting point**: the audio side and the text side each learn a route to it during training, and at test time you only need the text route.

## 3. Key Concepts

- **End-to-end (text → waveform).** No intermediate mel + no separately trained vocoder. The decoder *is* a HiFi-GAN generator trained jointly, so there's no acoustic-model/vocoder mismatch.
- **Conditional VAE (CVAE).** The training objective is an ELBO: a **reconstruction** term (the decoder must rebuild the audio from `z`) plus a **KL** term that pulls the audio-derived posterior `q(z|audio)` toward the text-conditioned prior `p(z|text)`.
- **Posterior encoder.** Reads a **linear-scale spectrogram** of the target audio and outputs the mean/variance of `q(z|audio)`. Used only in training.
- **Normalizing flow.** An invertible network `f_θ` that makes the simple text-conditioned prior far more expressive, so a Gaussian-ish prior can match a complex posterior. Invertible → exact likelihood, and you run it **backwards** at inference.
- **Monotonic Alignment Search (MAS).** A dynamic-programming search for the **best monotonic, non-skipping** alignment between text tokens and audio frames — a *hard* alignment learned without external aligners. This replaces Tacotron's fragile soft attention.
- **Stochastic Duration Predictor (SDP).** A flow-based module that **samples** how many frames each token occupies, conditioned on text. This produces natural rhythm variation (one-to-many) instead of a single fixed cadence.
- **Length regulator / expansion.** Given per-token durations, each token's encoding is repeated to span its frames — turning a text-length sequence into an audio-frame-length sequence (non-autoregressive, fully parallel).
- **Adversarial training.** A multi-period/scale **discriminator** judges the generated waveform against real audio (plus a feature-matching loss), pushing the decoder toward realistic high-frequency detail — inherited from HiFi-GAN.
- **Reparameterization trick.** `z = μ + σ ⊙ ε`, `ε ~ N(0, I)` — lets gradients flow through the sampling step in the VAE.

## 4. Setup

The two worked examples below illustrate the core algorithms (**MAS** and **stochastic duration → length regulation**) with **NumPy only**, so they run on any CPU in milliseconds. Real synthesis (Example 3) is gated behind an env var and uses 🤗 Transformers' `VitsModel`.

```bash
%pip install numpy           # for the two conceptual examples
%pip install transformers torch   # only for the gated real-synthesis cell
```

Ways to actually run VITS today:

- **🤗 Transformers** — `from transformers import VitsModel, AutoTokenizer`; e.g. `facebook/mms-tts-eng` (Meta MMS, VITS-based, 1100+ languages). Used, gated, in Example 3.
- **Coqui TTS** — `pip install TTS`, then `tts --model_name tts_models/en/ljspeech/vits ...`. See [[coqui-tts]].
- **Official repo** — `jaywalnut310/vits` (the reference implementation).

In [1]:
import numpy as np

# The two conceptual examples use only NumPy. Transformers/torch are optional and
# only needed for the gated real-synthesis cell (Example 3).
print("numpy:", np.__version__)

try:
    import transformers
    print("transformers:", transformers.__version__, "(real synthesis available)")
except Exception as e:
    transformers = None
    print("transformers not installed:", type(e).__name__,
          "-> NumPy examples still run; Example 3 prints the call shape only.")

numpy: 2.5.0


/Users/danieldekerlegand/Development/ai-tutor/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers: 5.12.1 (real synthesis available)


## 5. Worked Examples

### Example 1 — Monotonic Alignment Search (MAS)

MAS is the heart of VITS training: given a per-(frame, token) score matrix (in VITS, the log-likelihood of each audio frame's latent under each text token's prior), it finds the **single best monotonic alignment** — frames advance in time, the token index never goes backward and never skips a token. The result tells us **how many frames each token gets** (its duration), which is exactly what the duration predictor is later trained to reproduce. Here we implement the DP from scratch on a tiny synthetic score matrix.

In [2]:
# Example 1 - Monotonic Alignment Search via dynamic programming.
def monotonic_alignment_search(value):
    """value[t, s] = score of assigning audio frame t to text token s.
    Returns the most-likely monotonic, non-skipping path token-index per frame."""
    T, S = value.shape                      # T audio frames, S text tokens
    Q = np.full((T, S), -1e9)               # best cumulative score to reach (t, s)
    Q[0, 0] = value[0, 0]                    # frame 0 must align to token 0
    for t in range(1, T):
        for s in range(min(t + 1, S)):      # token s reachable only if enough frames passed
            stay = Q[t - 1, s]              # frame t stays on same token as frame t-1
            advance = Q[t - 1, s - 1] if s > 0 else -1e9  # moved on to a new token
            Q[t, s] = value[t, s] + max(stay, advance)

    # Backtrack from the corner (last frame must land on the last token).
    path = np.zeros(T, dtype=int)
    s = S - 1
    for t in range(T - 1, -1, -1):
        path[t] = s
        if t > 0 and s > 0 and Q[t - 1, s - 1] >= Q[t - 1, s]:
            s -= 1                           # the cheaper way in was via the previous token
    return path


rng = np.random.default_rng(0)
tokens = list("h e l o")[::2]               # 4 text tokens: ['h','e','l','o']
T, S = 12, len(tokens)                       # 12 audio frames, 4 tokens

# Build a score matrix that secretly favours a diagonal-ish alignment, plus noise.
true_centers = np.linspace(0, S - 1, T)
value = -((np.arange(S)[None, :] - true_centers[:, None]) ** 2)   # high near the diagonal
value += 0.3 * rng.standard_normal((T, S))                        # realistic noise

path = monotonic_alignment_search(value)
print("token index per frame:", path.tolist())
print("monotonic (never goes backward)?", bool(np.all(np.diff(path) >= 0)))
print("every token used?", sorted(set(path)) == list(range(S)))

durations = np.bincount(path, minlength=S)   # frames per token -> the duration target
for tok, d in zip(tokens, durations):
    print(f"  token {tok!r}: {d} frames  {'█' * d}")
print("total frames:", durations.sum(), "== T?", durations.sum() == T)

token index per frame: [0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3]
monotonic (never goes backward)? True
every token used? True
  token 'h': 3 frames  ███
  token 'e': 3 frames  ███
  token 'l': 3 frames  ███
  token 'o': 3 frames  ███
total frames: 12 == T? True


### Example 2 — Stochastic duration → length regulation (the one-to-many trick)

At inference VITS doesn't use MAS — instead the **stochastic duration predictor** *samples* a duration per token, so the same text gets different, natural rhythms each time. Those durations drive the **length regulator**, which repeats each token's encoding to fill its frames, turning a token-length sequence into a frame-length one (fully parallel, no autoregression). Below we sample durations twice and watch the rhythm — and total length — change. We also show the VAE **reparameterization** that produces the latent `z` fed to the decoder.

In [3]:
# Example 2 - sample stochastic durations, length-regulate, reparameterize z.
rng = np.random.default_rng(42)
tokens = list("hello")
D = 4                                         # encoder hidden dim per token
text_enc = rng.standard_normal((len(tokens), D)).astype(np.float32)  # encoder output

# A deterministic model would emit one fixed duration per token. The stochastic
# duration predictor instead samples around a base rate -> rhythm varies per run.
base = np.array([2, 3, 2, 2, 4])             # mean frames per token (longer for 'o')

def sample_durations(base, temperature=1.0):
    noise = temperature * rng.standard_normal(len(base))
    dur = np.maximum(1, np.round(base * np.exp(0.25 * noise))).astype(int)  # >=1 frame
    return dur

def length_regulate(enc, dur):
    """Repeat each token's encoding `dur[i]` times -> frame-length sequence."""
    return np.repeat(enc, dur, axis=0)

for take in (1, 2):
    dur = sample_durations(base)
    frames = length_regulate(text_enc, dur)
    rhythm = " ".join(f"{t}:{d}" for t, d in zip(tokens, dur))
    print(f"take {take}: durations {rhythm}  -> {frames.shape[0]} frames, "
          f"expanded enc shape {frames.shape}")

# Reparameterization trick: the decoder consumes z = mu + sigma * eps.
T = frames.shape[0]
mu = rng.standard_normal((T, D)).astype(np.float32)        # prior mean (from text+flow)
log_sigma = -0.5 * np.ones((T, D), dtype=np.float32)
eps = rng.standard_normal((T, D)).astype(np.float32)
z = mu + np.exp(log_sigma) * eps
print(f"\nlatent z fed to the decoder: shape {z.shape}, dtype {z.dtype}")
print("the HiFi-GAN-style decoder turns this z directly into a waveform - no mel step")

take 1: durations h:2 e:3 l:3 l:2 o:4  -> 14 frames, expanded enc shape (14, 4)
take 2: durations h:2 e:3 l:2 l:2 o:4  -> 13 frames, expanded enc shape (13, 4)

latent z fed to the decoder: shape (13, 4), dtype float32
the HiFi-GAN-style decoder turns this z directly into a waveform - no mel step


### Example 3 — Real synthesis with 🤗 Transformers `VitsModel` (gated)

This needs `transformers` + `torch` and a one-time model download (~145 MB for `facebook/mms-tts-eng`), so it runs only when `RUN_VITS=1`. Otherwise it prints the exact call shape. Note there is **one** model call: tokens → waveform, with no separate vocoder step.

In [4]:
# Example 3 - end-to-end synthesis (gated behind RUN_VITS + transformers + download).
import os

ready = os.getenv("RUN_VITS") and transformers is not None

if ready:
    import torch
    from transformers import VitsModel, AutoTokenizer

    model = VitsModel.from_pretrained("facebook/mms-tts-eng").eval()
    tok = AutoTokenizer.from_pretrained("facebook/mms-tts-eng")

    inputs = tok("Hello world, this is VITS speaking.", return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs).waveform        # (1, num_samples) - audio, in one call
    wav = out[0].cpu().numpy()
    sr = model.config.sampling_rate
    print(f"waveform: {wav.shape}  ~{wav.shape[-1] / sr:.2f}s @ {sr} Hz")
    # import scipy.io.wavfile as wf; wf.write('out.wav', sr, wav)
else:
    print("RUN_VITS not set (or transformers missing) - showing the call shape:\n")
    print("  from transformers import VitsModel, AutoTokenizer")
    print("  model = VitsModel.from_pretrained('facebook/mms-tts-eng').eval()")
    print("  tok   = AutoTokenizer.from_pretrained('facebook/mms-tts-eng')")
    print("  inputs = tok('Hello world.', return_tensors='pt')")
    print("  wav = model(**inputs).waveform   # text tokens -> waveform, ONE model")

RUN_VITS not set (or transformers missing) - showing the call shape:

  from transformers import VitsModel, AutoTokenizer
  model = VitsModel.from_pretrained('facebook/mms-tts-eng').eval()
  tok   = AutoTokenizer.from_pretrained('facebook/mms-tts-eng')
  inputs = tok('Hello world.', return_tensors='pt')
  wav = model(**inputs).waveform   # text tokens -> waveform, ONE model


## 6. Gotchas & Pitfalls

- **Output is non-deterministic by design.** The stochastic duration predictor (and prior sampling) mean the *same text yields different audio each run*. Great for naturalness, surprising in tests — seed the RNG, or expose a noise-scale / `noise_scale_w` knob if you need reproducibility.
- **Front-end (phonemes) still matters.** VITS removes the *vocoder* seam, not text normalization. Many strong VITS checkpoints expect **phonemes** (e.g. via `espeak-ng`/`phonemizer`); feeding raw text to a phoneme-trained model degrades quality. Check what the checkpoint's tokenizer expects. See [[espeak-ng]], [[phoneme-analysis]].
- **Training is heavier and trickier than two-stage.** Joint CVAE + flow + GAN + MAS means more moving parts, GAN instability, and longer training than a Tacotron-style acoustic model alone. Mismatched STFT/spectrogram configs between the posterior encoder and your data are a classic silent-quality bug.
- **Single-speaker by default.** Vanilla VITS is one voice; multi-speaker needs speaker embeddings (the paper's multi-speaker variant / VCTK models). It is **not** a zero-shot voice cloner — use [[styletts2]] / [[bark]] for that.
- **“No vocoder” ≠ “no spectrogram anywhere.”** The *posterior encoder* still consumes a **linear** spectrogram of the target audio at **training** time. The point is that inference is mel-free and single-model, not that spectrograms never appear.
- **MAS is train-only.** Don't expect an attention/alignment plot at inference — alignment is replaced by the sampled durations. Debugging bad rhythm means inspecting the duration predictor, not an attention heatmap.
- **Long inputs.** Like most TTS, very long single utterances strain prosody/length prediction; split into sentences and synthesize per-sentence.

## 7. When to Use vs Alternatives

| Option | Trade-off vs VITS |
|---|---|
| **[[tacotron]] + vocoder** | Two-stage, autoregressive, fragile attention, and a train/inference mel mismatch. Easier to understand and hack, but slower and lower-quality end-to-end than VITS. |
| **FastSpeech 2 + vocoder** | Non-autoregressive and fast like VITS, but still **two models** and needs external duration/pitch supervision. VITS learns alignment internally (MAS) and is single-stage. |
| **[[styletts2]]** | Style-diffusion TTS with strong expressiveness and zero-shot cloning — beats VITS on controllability/voice cloning, at the cost of a heavier, more complex system. |
| **[[bark]]** | Generates very expressive speech (laughs, music, non-verbal) and does cloning, but is a large autoregressive token model: slower, less controllable timing, no clean text→audio determinism. |
| **[[coqui-tts]]** | A framework that *packages* VITS (and others) with pretrained weights — the easiest way to actually run VITS. |
| **Cloud APIs ([[elevenlabs]], [[azure-speech]], [[amazon-polly]], [[google-cloud-tts]])** | Turnkey, scalable, no GPUs or training; but pay per character, less control, and you don't own the weights. |

**Choose VITS when** you want top-tier open-source single/multi-speaker TTS in one fast, jointly-trained model with natural prosody variation. **Choose something else** when you need zero-shot cloning or rich style control ([[styletts2]], [[bark]], [[elevenlabs]]), or a no-ops managed API (cloud).

## 8. Resources

- **Paper** — *Conditional Variational Autoencoder with Adversarial Learning for End-to-End Text-to-Speech* (Kim, Kong & Son, 2021): https://arxiv.org/abs/2106.06103
- **Official implementation** (`jaywalnut310/vits`): https://github.com/jaywalnut310/vits
- **Audio samples** from the authors: https://jaywalnut310.github.io/vits-demo/index.html
- **🤗 Transformers `VitsModel` docs** (used in Example 3): https://huggingface.co/docs/transformers/model_doc/vits
- **Meta MMS-TTS** — VITS-based TTS for 1100+ languages: https://huggingface.co/facebook/mms-tts-eng
- **Coqui TTS** — practical pretrained VITS models: https://github.com/coqui-ai/TTS